In [1]:
import requests
import pandas as pd
import time

# 1. 基础配置
AMAP_KEY = "0e971d578b21c1bdb0e07daf1650576b"
CITY = "北京"
KEYWORDS = ["蜜雪冰城"]  # 关键词，高德会根据这些搜
OUTPUT_FILE = "beijing_drink_shops_amap.csv"

def get_poi_data(keyword: str, city: str, key: str) -> list:
    """从高德获取单一关键词的所有POI数据"""
    all_pois = []
    page = 1
    
    while True:
        # 高德POI搜索接口地址 (Web服务 API)
        url = f"https://restapi.amap.com/v3/place/text?key={key}&keywords={keyword}&city={city}&page={page}&offset=20"
        
        try:
            response = requests.get(url, timeout=10)
            data = response.json()
            
            if data['status'] == '1':
                pois = data['pois']
                if not pois: # 抓完了，没下一页了
                    break
                
                for poi in pois:
                    # 提取我们需要的信息
                    item = {
                        'name': poi['name'],
                        'type': poi['type'],
                        'address': poi['address'],
                        'lon': poi['location'].split(',')[0], # 高德坐标是 "经度,纬度" 字符串
                        'lat': poi['location'].split(',')[1],
                        'pname': poi['pname'],      # 省份
                        'cityname': poi['cityname'], # 城市
                        'adname': poi['adname']      # 区县
                    }
                    all_pois.append(item)
                
                print(f"正在抓取【{keyword}】第 {page} 页，共抓到 {len(all_pois)} 条...")
                page += 1
                
                # 高德API对普通开发者有频率限制，休息一下，防止被封
                time.sleep(0.1) 
            else:
                print(f"❌ 抓取失败: {data['info']}")
                break
        except Exception as e:
            print(f"⚠️ 请求出错: {e}")
            break
            
    return all_pois

# 2. 执行抓取
final_list = []
for kw in KEYWORDS:
    print(f"--- 🚀 开始抓取关键词: {kw} ---")
    results = get_poi_data(kw, CITY, AMAP_KEY)
    final_list.extend(results)

# 3. 数据去重与保存
if final_list:
    df = pd.DataFrame(final_list)
    # 按照名称和位置去重（因为“奶茶”和“饮品店”可能重复搜出同一家店）
    df = df.drop_duplicates(subset=['name', 'lon', 'lat'])
    
    print(f"\n✅ 抓取结束！北京全城共发现 {len(df)} 家唯一饮品店。")
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"💾 数据已成功存入: {OUTPUT_FILE}")
else:
    print("😢 抓取失败，没拿到任何数据，请检查API Key状态。")


--- 🚀 开始抓取关键词: 蜜雪冰城 ---
正在抓取【蜜雪冰城】第 1 页，共抓到 20 条...
正在抓取【蜜雪冰城】第 2 页，共抓到 40 条...
正在抓取【蜜雪冰城】第 3 页，共抓到 60 条...
正在抓取【蜜雪冰城】第 4 页，共抓到 80 条...
正在抓取【蜜雪冰城】第 5 页，共抓到 100 条...
正在抓取【蜜雪冰城】第 6 页，共抓到 120 条...
正在抓取【蜜雪冰城】第 7 页，共抓到 140 条...
正在抓取【蜜雪冰城】第 8 页，共抓到 160 条...
正在抓取【蜜雪冰城】第 9 页，共抓到 180 条...
正在抓取【蜜雪冰城】第 10 页，共抓到 200 条...
正在抓取【蜜雪冰城】第 11 页，共抓到 220 条...
正在抓取【蜜雪冰城】第 12 页，共抓到 240 条...
正在抓取【蜜雪冰城】第 13 页，共抓到 260 条...
正在抓取【蜜雪冰城】第 14 页，共抓到 280 条...
正在抓取【蜜雪冰城】第 15 页，共抓到 300 条...
正在抓取【蜜雪冰城】第 16 页，共抓到 320 条...
正在抓取【蜜雪冰城】第 17 页，共抓到 340 条...
正在抓取【蜜雪冰城】第 18 页，共抓到 360 条...
正在抓取【蜜雪冰城】第 19 页，共抓到 380 条...
正在抓取【蜜雪冰城】第 20 页，共抓到 400 条...
正在抓取【蜜雪冰城】第 21 页，共抓到 420 条...
正在抓取【蜜雪冰城】第 22 页，共抓到 440 条...
正在抓取【蜜雪冰城】第 23 页，共抓到 460 条...
正在抓取【蜜雪冰城】第 24 页，共抓到 473 条...

✅ 抓取结束！北京全城共发现 473 家唯一饮品店。
💾 数据已成功存入: beijing_drink_shops_amap.csv
